# BirdCLEF+ 2026

## Score: .768

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
from scipy.ndimage import zoom
import librosa
from pathlib import Path
from tqdm import tqdm
import ast

DATA_DIR = Path("/kaggle/input/competitions/birdclef-2026")

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
taxonomy_df = pd.read_csv(DATA_DIR / "taxonomy.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
soundscape_labels = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")

SPECIES = [c for c in sample_sub.columns if c != "row_id"]
NUM_CLASSES = len(SPECIES)
species_to_idx = {s: i for i, s in enumerate(SPECIES)}

print(f"Species: {NUM_CLASSES}")
print(f"Train audio: {len(train_df)}")
print(f"Train soundscape labels: {len(soundscape_labels)}")

In [ ]:
SR = 32000
DURATION = 5.0
N_MELS, N_FFT, HOP_LEN = 128, 2048, 512
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 12
LR = 1e-3
DEVICE = "cpu"
MAX_SAMPLES = 20000

print(f"Device: {DEVICE}")

In [ ]:
def parse_start_seconds(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return 0
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).strip()
    if ":" in s:
        parts = s.split(":")
        return sum(float(p) * (60 ** (len(parts) - 1 - i)) for i, p in enumerate(parts))
    return float(s)


def load_audio_segment(filepath, sr=SR, duration=DURATION, start=0):
    start = parse_start_seconds(start)
    y, _ = librosa.load(filepath, sr=sr, mono=True)
    target_len = int(sr * duration)
    if start > 0:
        start_sample = int(start * sr)
        y = y[start_sample : start_sample + target_len]
    elif len(y) >= target_len:
        start_sample = (len(y) - target_len) // 2
        y = y[start_sample : start_sample + target_len]
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode="constant")
    return y[:target_len]


def audio_to_mel(y, n_mels=N_MELS, n_fft=N_FFT, hop_len=HOP_LEN, fmin=0, fmax=16000):
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=n_mels, n_fft=n_fft, hop_length=hop_len, fmin=fmin, fmax=fmax)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    h, w = mel_db.shape
    zoom_factors = (IMG_SIZE[0] / h, IMG_SIZE[1] / w)
    mel_resized = zoom(mel_db, zoom_factors, order=1)
    return mel_resized.astype(np.float32)


def parse_labels(labels_str):
    if pd.isna(labels_str) or labels_str == "":
        return []
    return [s.strip() for s in str(labels_str).split(";") if s.strip()]


def parse_secondary(sec_str):
    if pd.isna(sec_str) or sec_str == "":
        return []
    s = str(sec_str).strip()
    if s.startswith("["):
        try:
            return ast.literal_eval(s)
        except:
            return parse_labels(s.replace("[", "").replace("]", "").replace("'", ""))
    return parse_labels(s)

In [ ]:
def spec_augment(mel, freq_mask_param=16, time_mask_param=32):
    mel = mel.copy()
    n_mels, n_frames = mel.shape
    if np.random.random() < 0.6:
        f = min(freq_mask_param, n_mels)
        f0 = np.random.randint(0, max(1, n_mels - f))
        mel[f0:f0 + f, :] = mel.mean()
    if np.random.random() < 0.6:
        t = min(time_mask_param, n_frames)
        t0 = np.random.randint(0, max(1, n_frames - t))
        mel[:, t0:t0 + t] = mel.mean()
    return mel


def mixup(x1, x2, y1, y2, alpha=0.2):
    lam = np.random.beta(alpha, alpha)
    x = lam * x1 + (1 - lam) * x2
    y = lam * y1 + (1 - lam) * y2
    return x, y

In [ ]:
class BirdAudioDataset(Dataset):
    def __init__(self, df, audio_dir, species_to_idx, use_secondary=True, augment=True):
        self.df = df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.species_to_idx = species_to_idx
        self.use_secondary = use_secondary
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def _labels_to_multihot(self, row):
        labels = set(parse_labels(row.get("primary_label", "")))
        if self.use_secondary and "secondary_labels" in row:
            labels.update(parse_secondary(row["secondary_labels"]))
        y = np.zeros(NUM_CLASSES, dtype=np.float32)
        for sp in labels:
            if sp in self.species_to_idx:
                y[self.species_to_idx[sp]] = 1.0
        return y

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filepath = self.audio_dir / row["filename"]
        start = row.get("start", 0)
        y_audio = load_audio_segment(str(filepath), start=start)
        mel = audio_to_mel(y_audio)
        mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-8)
        if self.augment and np.random.random() < 0.6:
            mel = spec_augment(mel)
        x = torch.from_numpy(mel).unsqueeze(0).repeat(3, 1, 1).float()  # (3, H, W) per sample
        y = torch.from_numpy(self._labels_to_multihot(row))
        return x, y

In [ ]:
class EfficientNetClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        try:
            w = tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = tv_models.efficientnet_b0(weights=w)
        except Exception:
            self.backbone = tv_models.efficientnet_b0(weights=None)
        self.backbone.classifier[1] = nn.Linear(self.backbone.classifier[1].in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

In [ ]:
train_audio_df = train_df.copy()
train_audio_df["start"] = 0
train_audio_df = train_audio_df[["filename", "primary_label", "secondary_labels", "start"]].copy()
train_audio_df["filename"] = "train_audio/" + train_audio_df["filename"]

ss_df = soundscape_labels.copy()
ss_df["secondary_labels"] = ""
ss_df["filename"] = "train_soundscapes/" + ss_df["filename"].astype(str).str.replace(".ogg", "", regex=False) + ".ogg"
ss_df = ss_df[["filename", "primary_label", "secondary_labels", "start"]].copy()

combined_df = pd.concat([train_audio_df, ss_df], ignore_index=True)
combined_df = combined_df[combined_df["filename"].apply(lambda f: (DATA_DIR / f).exists())]
if MAX_SAMPLES is not None and len(combined_df) > MAX_SAMPLES:
    counts = combined_df["primary_label"].value_counts()
    inv_freq = combined_df["primary_label"].map(lambda x: 1.0 / counts.get(x, 1))
    combined_df = combined_df.sample(n=MAX_SAMPLES, weights=inv_freq, random_state=42).reset_index(drop=True)
    print(f"Sampled to {len(combined_df)} samples (MAX_SAMPLES={MAX_SAMPLES})")
print(f"Combined train samples: {len(combined_df)}")

In [ ]:
def collate_mixup(batch):
    xs, ys = zip(*batch)
    xs, ys = list(xs), list(ys)
    if np.random.random() < 0.35 and len(xs) >= 2:
        i, j = np.random.choice(len(xs), 2, replace=False)
        xm, ym = mixup(xs[i], xs[j], ys[i], ys[j])
        xs[i], ys[i] = xm, ym
    return torch.stack(xs), torch.stack(ys)


train_ds = BirdAudioDataset(combined_df, DATA_DIR, species_to_idx, use_secondary=True, augment=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_mixup)

model = EfficientNetClassifier(NUM_CLASSES, pretrained=True).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
Path("models").mkdir(exist_ok=True)
torch.save({"model_state_dict": model.state_dict(), "epoch": EPOCHS}, "models/best.pt")

In [ ]:
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1} loss: {total_loss/len(train_loader):.4f}")

In [ ]:
def merge_windows_to_chunks(probs_windows, starts, seg_len, n_chunks):
    merged = np.zeros((n_chunks, len(SPECIES)), dtype=np.float64)
    counts = np.zeros((n_chunks, 1), dtype=np.float64)
    for j, s in enumerate(starts):
        i_lo = max(0, s // seg_len)
        i_hi = min(n_chunks - 1, (s + seg_len - 1) // seg_len)
        for i in range(i_lo, i_hi + 1):
            merged[i] += probs_windows[j]
            counts[i] += 1
    return (merged / np.maximum(counts, 1)).astype(np.float32)

# Parse expected row_ids: "stem_endsec" -> stem -> [end_sec]
import re
row_pattern = re.compile(r"^(.*)_(\d+)$")
def parse_row_id(rid):
    m = row_pattern.match(str(rid))
    return (m.group(1), int(m.group(2))) if m else (None, None)

expected_ids = sample_sub["row_id"].tolist()
expected_by_stem = {}
for rid in expected_ids:
    stem, end_sec = parse_row_id(rid)
    if stem:
        expected_by_stem.setdefault(stem, []).append(end_sec)
for stem in expected_by_stem:
    expected_by_stem[stem] = sorted(expected_by_stem[stem])

# File discovery: test_soundscapes first, fallback to train_soundscapes (LB kernel style)
test_dir = DATA_DIR / "test_soundscapes"
train_sc_dir = DATA_DIR / "train_soundscapes"
soundscape_files = sorted(test_dir.glob("*.ogg")) if test_dir.exists() else []
if not soundscape_files and expected_by_stem:
    for stem in sorted(expected_by_stem):
        p = train_sc_dir / f"{stem}.ogg"
        if p.exists():
            soundscape_files.append(p)
    if soundscape_files:
        print(f"No test .ogg; using {len(soundscape_files)} train_soundscapes.")
print(f"Soundscape files: {len(soundscape_files)}")

model.eval()
seg_len = int(SR * DURATION)
stride = seg_len // 2
all_row_ids, all_preds = [], []
with torch.no_grad():
    for fpath in tqdm(soundscape_files, desc="Predicting"):
        fname = fpath.stem
        y, _ = librosa.load(str(fpath), sr=SR, mono=True)
        # Use expected end_secs to determine n_chunks (LB kernel style)
        if fname in expected_by_stem:
            n_chunks = max(expected_by_stem[fname]) // int(DURATION)
        else:
            n_chunks = max(1, len(y) // seg_len)
        padded_len = n_chunks * seg_len
        padded = np.pad(y, (0, max(0, padded_len - len(y))))[:padded_len]
        starts = list(range(0, len(padded) - seg_len + 1, stride))
        probs_list = []
        for s in starts:
            seg = padded[s:s + seg_len]
            mel = audio_to_mel(seg)
            mel = (mel - mel.min()) / (mel.max() - mel.min() + 1e-8)
            x = torch.from_numpy(mel).unsqueeze(0).repeat(1, 3, 1, 1).float().to(DEVICE)
            logits = model(x)
            probs_list.append(torch.sigmoid(logits).cpu().numpy()[0])
        probs_windows = np.array(probs_list)
        probs = merge_windows_to_chunks(probs_windows, starts, seg_len, n_chunks)
        if n_chunks > 4:
            sharp = np.power(probs, 1.5)
            w = np.array([0.05, 0.15, 0.60, 0.15, 0.05])
            pad = np.pad(sharp, ((2, 2), (0, 0)), mode="edge")
            smooth = w[0]*pad[:-4] + w[1]*pad[1:-3] + w[2]*pad[2:-2] + w[3]*pad[3:-1] + w[4]*pad[4:]
            probs = np.power(smooth, 1/1.5)
        elif n_chunks > 2:
            w = np.array([0.20, 0.60, 0.20])
            pad = np.pad(probs, ((1, 1), (0, 0)), mode="edge")
            probs = w[0]*pad[:-2] + w[1]*pad[1:-1] + w[2]*pad[2:]
        file_max = np.max(probs, axis=0, keepdims=True) * 0.05
        probs = probs + file_max
        # Only output chunks in expected (LB kernel style)
        if fname in expected_by_stem:
            valid = [(e // int(DURATION)) - 1 for e in expected_by_stem[fname]]
            valid = [i for i in valid if 0 <= i < n_chunks]
        else:
            valid = list(range(n_chunks))
        for i in valid:
            all_row_ids.append(f"{fname}_{(i+1)*int(DURATION)}")
            all_preds.append(probs[i])

if all_preds:
    sub = pd.DataFrame(all_preds, columns=SPECIES, index=all_row_ids)
    sub = sub[~sub.index.duplicated(keep="first")]
    if (sc_path := DATA_DIR / "train_soundscapes_labels.csv").exists():
        sc_labels = pd.read_csv(sc_path)
        species_counts = {s: 0 for s in SPECIES}
        for row in sc_labels["primary_label"].dropna():
            for sp in str(row).split(";"):
                sp = sp.strip()
                if sp in species_counts:
                    species_counts[sp] += 1
        total_seg = max(len(sc_labels), 1)
        base_rates = np.array([species_counts[s] / total_seg for s in SPECIES], dtype=np.float32)
        prior_mask = (base_rates == 0).astype(np.float32)
        suppression = 1.0 - 0.15 * prior_mask
        sub[SPECIES] = sub[SPECIES].values * suppression
    sub[SPECIES] = sub[SPECIES].clip(0, 1)
    sub = sub.reindex(expected_ids).fillna(1.0 / NUM_CLASSES)
    sub = sub.reset_index().rename(columns={"index": "row_id"})
    sub = sub[["row_id"] + SPECIES]
else:
    sub = sample_sub.copy()
    sub[SPECIES] = 1.0 / NUM_CLASSES
sub.to_csv("submission.csv", index=False)
print(sub.head())